In [2]:
#pip install statsmodels

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   - -----------------------------


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, mean_squared_error, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
import statsmodels.api as sm # You might use this for statistical testing later
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

In [2]:
# --- Main Execution ---

# Load your data (replace with your actual file path)
# 1. Load the Data
excel_file = "Garmin_activities.xlsx"
sheet_name = "Garmin_data"

data = pd.read_excel(excel_file, sheet_name=sheet_name)
data


,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Avg Resp,Min Resp,Max Resp,Moving Time,Elapsed Time,Min Elevation,Max Elevation,userID,location,Series totales
0,Gym training,2022-02-06 19:16:03.993,False,Fuerza,0.00,131.0,1899-12-29 00:44:45,99.0,134.0,NaN,...,NaN,NaN,NaN,0.031076,0.031076,NaN,NaN,AMG,FUERZA,--
1,Gym training,2022-02-22 20:36:51.983,False,Fuerza,0.00,116.0,1899-12-29 00:32:37,111.0,134.0,NaN,...,NaN,NaN,NaN,0.022650,0.022650,NaN,NaN,AMG,FUERZA,--
2,Gym training,2022-06-15 11:39:56.016,False,Fuerza,0.00,90.0,1899-12-29 00:34:18,95.0,121.0,NaN,...,NaN,NaN,NaN,0.023819,0.023819,NaN,NaN,AMG,FUERZA,--
3,Indoor Cycling,2022-09-02 10:03:48.036,False,Ciclismo en sala,20.71,350.0,1899-12-29 00:56:26,146.0,177.0,NaN,...,NaN,NaN,NaN,0.039155,0.039190,NaN,NaN,AMG,INDOOR,--
4,Indoor Cycling,2022-09-26 11:09:46.023,False,Ciclismo en sala,17.77,305.0,1899-12-29 00:46:51,151.0,178.0,NaN,...,NaN,NaN,NaN,0.032465,0.035579,NaN,NaN,AMG,INDOOR,--
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1128,Road Cycling,2022-08-27 07:13:23.980,False,Tenjo Road Cycling,115.11,2277.0,1899-12-29 05:15:36,134.0,184.0,45.0,...,27.0,13.0,40.0,0.218576,0.295498,1663.0,2759.0,AARL,TENJO,NaN
1129,Road Cycling,2022-09-04 06:49:00.970,False,Tenjo Road Cycling,129.91,2995.0,1899-12-29 05:46:22,141.0,181.0,50.0,...,30.0,13.0,42.0,0.240231,0.279259,1185.0,2748.0,AARL,TENJO,NaN
1130,Road Cycling,2022-03-06 08:28:23.003,False,Villeta Road Cycling,81.81,2335.0,1899-12-29 05:41:49,155.0,183.0,50.0,...,34.0,19.0,43.0,0.232662,0.366759,756.0,2823.0,KMP,VILLETA,NaN
1131,Road Cycling,2022-03-06 08:31:53.990,False,Villeta Ciclismo en ruta,105.91,3433.0,1899-12-29 07:12:14,137.0,172.0,50.0,...,26.0,10.0,40.0,0.273171,0.401157,721.0,2787.0,AARL,VILLETA,NaN


In [4]:
# --- Data Cleaning (example) ---
# Data Cleaning and Preparation (Improved)

# Convert 'Date' to datetime objects (if needed)
data['Date'] = pd.to_datetime(data['Date'], errors='coerce') # Handle potential errors in date conversion

# Handle missing values (example: fill with mean for numeric columns)
for col in data.select_dtypes(include=np.number).columns:
    data[col].fillna(data[col].mean(), inplace=True)

# Clean 'Time' column by replacing '####' with NaN and then imputing with a default value
# Check if 'Time' column exists before cleaning
if 'Time' in data.columns:
    data['Time'] = data['Time'].replace('####', pd.NA)
    data['Time'] = data['Time'].fillna('00:00:00')  # Replace NaN with a default time
else:
    print("Warning: 'Time' column not found in the dataset.")

# Drop duplicates
data.drop_duplicates(inplace=True)


# --- Feature Engineering (example) ---
# Create a simple 'hour_of_day' feature
data['hour_of_day'] = data['Date'].dt.hour


C:\Users\Alejandro.rodriguezl\AppData\Local\Temp\ipykernel_40696\2112270659.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].mean(), inplace=True)


In [6]:
# --- 1. Predicting Activity Outcome (Distance) ---

def predict_activity_outcome(data, target_variable='Distance'):
    """
    Predicts activity outcome (Distance) using regression models.

    Args:
        data (pd.DataFrame): DataFrame containing Garmin activity data.
        target_variable (str): 'Distance' or other numerical variable to predict.

    Returns:
        None: Prints model results.
    """

    # Features (add more as needed)
    features = ['Distance', 'Avg HR', 'Total Ascent', 'Total Descent', 'Calories', 'Max Speed']  # Example features, added Calories and Max Speed

    # Preprocessing
    data = data[features + [target_variable]].copy()
    data.replace([np.inf, -np.inf], np.nan, inplace=True)  # Handle infinities

    # Imputation: Handle missing values explicitly for each feature
    for feature in features + [target_variable]:
        if data[feature].isnull().any():
            if pd.api.types.is_numeric_dtype(data[feature]):
                imputer = SimpleImputer(strategy='mean')  # Or 'median'
                data[feature] = imputer.fit_transform(data[[feature]]).ravel()
            else:
                imputer = SimpleImputer(strategy='most_frequent')
                data[feature] = imputer.fit_transform(data[[feature]]).ravel()

    X = data[features]
    y = data[target_variable].values.ravel()  # Ensure y is 1D array

    # Scaling
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split data with cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold cross-validation

    # --- Linear Regression ---
    model_lr = LinearRegression()
    mse_scores_lr = -cross_val_score(model_lr, X, y, cv=kf, scoring='neg_mean_squared_error')
    print(f"Linear Regression Mean MSE: {mse_scores_lr.mean()}")

    # --- Random Forest Regression ---
    model_rf = RandomForestRegressor(random_state=42)
    mse_scores_rf = -cross_val_score(model_rf, X, y, cv=kf, scoring='neg_mean_squared_error')
    print(f"Random Forest Regression Mean MSE: {mse_scores_rf.mean()}")

    # --- Gradient Boosting Regression ---
    model_gb = GradientBoostingRegressor(random_state=42)
    mse_scores_gb = -cross_val_score(model_gb, X, y, cv=kf, scoring='neg_mean_squared_error')
    print(f"Gradient Boosting Regression Mean MSE: {mse_scores_gb.mean()}")

In [7]:
# Imputation: Handle missing values for 'Distance'
if data['Distance'].isnull().any():
    imputer = SimpleImputer(strategy='mean')  # Or 'median'
    data['Distance'] = imputer.fit_transform(data[['Distance']]).ravel()

In [26]:
# Call the functions
predict_activity_outcome(data, target_variable='Distance')

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [11]:
# --- 2. Performance Anomaly Detection ---

def detect_performance_anomalies(data):
    """
    Detects performance anomalies in Garmin data.

    Args:
        data (pd.DataFrame): DataFrame containing Garmin activity data.

    Returns:
        None: Prints anomaly detection results.
    """

    # Feature for anomaly detection (e.g., 'Avg Speed')
    anomaly_feature = 'Avg Speed'

    # Preprocessing
    data = data[[anomaly_feature]].copy()
    data.replace([np.inf, -np.inf], np.nan, inplace=True)  # Handle infinities
    
    # Imputation
    imputer = SimpleImputer(strategy='mean')
    data.fillna(data.mean(), inplace=True)
    data = imputer.fit_transform(data)

    # Scaling
    scaler = StandardScaler()
    data = scaler.fit_transform(data)

    # --- Isolation Forest ---
    model_if = IsolationForest(contamination='auto', random_state=42)
    model_if.fit(data)
    anomalies_if = model_if.predict(data)
    print("Isolation Forest Anomalies:", np.sum(anomalies_if == -1))  # -1 indicates anomaly

    # --- One-Class SVM ---
    model_svm = OneClassSVM(gamma='scale')
    model_svm.fit(data)
    anomalies_svm = model_svm.predict(data)
    print("One-Class SVM Anomalies:", np.sum(anomalies_svm == -1))  # -1 indicates anomaly



In [22]:
# Clean 'Avg Speed' column by removing non-numeric values
data['Avg Speed'] = pd.to_numeric(data['Avg Speed'], errors='coerce')

# Call the function
detect_performance_anomalies(data)


Isolation Forest Anomalies: 283
One-Class SVM Anomalies: 566


In [14]:
# --- 3. Predicting Risk of Injury ---

def predict_injury_risk(data):
    """
    Predicts the risk of injury based on training load and activity intensity.

    Args:
        data (pd.DataFrame): DataFrame containing Garmin activity data.

    Returns:
        None: Prints classification results (example).
    """
    # Features (example - you'll need to define these)
    features = ['Distance', 'Avg HR', 'Max HR', 'Total Ascent', 'Calories'] # Added 'Calories'
    data = data[features].copy()

    # Create a target variable (example: high_intensity_flag)
    data['high_intensity_flag'] = (data['Avg HR'] > 150).astype(int)  # Example threshold

    # Preprocessing (handle missing values, scaling, etc.)
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Imputation
    for col in data.columns:
        if data[col].isnull().any():  # Check if the column has any NaN values
            if pd.api.types.is_numeric_dtype(data[col]):
                imputer = SimpleImputer(strategy='mean')  # Or 'median'
                data[col] = imputer.fit_transform(data[[col]]).ravel()
            else:
                imputer = SimpleImputer(strategy='most_frequent')
                data[col] = imputer.fit_transform(data[[col]]).ravel()
    
    X = data.drop('high_intensity_flag', axis=1)
    y = data['high_intensity_flag'].values.ravel()

    # Scaling
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # --- Example: Logistic Regression (Classification) ---
    # from sklearn.linear_model import LogisticRegression
    model_lr = LogisticRegression()
    model_lr.fit(X_train, y_train)
    y_pred_lr = model_lr.predict(X_test)
    print("Logistic Regression Report:\n", classification_report(y_test, y_pred_lr))
    print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)}")

    # --- Example: Gaussian Naive Bayes (Classification) ---
    model_gnb = GaussianNB()
    model_gnb.fit(X_train, y_train)
    y_pred_gnb = model_gnb.predict(X_test)
    print("Gaussian Naive Bayes Report:\n", classification_report(y_test, y_pred_gnb))
    print(f"Gaussian Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_gnb)}")

    # --- Example: Decision Tree Classifier (Classification) ---
    model_dtc = DecisionTreeClassifier(random_state=42)
    model_dtc.fit(X_train, y_train)
    y_pred_dtc = model_dtc.predict(X_test)
    print("Decision Tree Classifier Report:\n", classification_report(y_test, y_pred_dtc))
    print(f"Decision Tree Classifier Accuracy: {accuracy_score(y_test, y_pred_dtc)}")



In [15]:
predict_injury_risk(data)


Logistic Regression Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99       180
           1       1.00      0.89      0.94        47

    accuracy                           0.98       227
   macro avg       0.99      0.95      0.97       227
weighted avg       0.98      0.98      0.98       227

Logistic Regression Accuracy: 0.9779735682819384
Gaussian Naive Bayes Report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.93       180
           1       0.78      0.68      0.73        47

    accuracy                           0.89       227
   macro avg       0.85      0.82      0.83       227
weighted avg       0.89      0.89      0.89       227

Gaussian Naive Bayes Accuracy: 0.8942731277533039
Decision Tree Classifier Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       180
           1       1.00      1.00      1.00  

In [17]:
# --- 4. User Behavior and Activity Patterns ---

def cluster_users(data):
    """
    Clusters users based on their activity patterns. This assumes you have
    a 'userID' column.

    Args:
        data (pd.DataFrame): DataFrame containing Garmin activity data.

    Returns:
        None: Prints clustering results.
    """
    # Features for clustering (example)
    cluster_features = ['Distance', 'Avg HR', 'Total Ascent', 'Calories'] # Added 'Calories'

    # Preprocessing
    user_data = data.groupby('userID')[cluster_features].mean()  # Aggregate data per user
    user_data.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Imputation
    imputer = SimpleImputer(strategy='mean')
    user_data = imputer.fit_transform(user_data)

    # Scaling
    scaler = StandardScaler()
    user_data_scaled = scaler.fit_transform(user_data)

    # --- K-Means Clustering ---
    kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')  # Example: 3 clusters
    kmeans.fit(user_data_scaled)
    labels_kmeans = kmeans.labels_
    print("K-Means Cluster Labels:", labels_kmeans)
    print("K-Means Silhouette Score:", silhouette_score(user_data_scaled, kmeans.labels_))

    # --- DBSCAN Clustering ---
    dbscan = DBSCAN(eps=0.5, min_samples=5)  # Example parameters
    labels_dbscan = dbscan.fit_predict(user_data_scaled)
    print("DBSCAN Cluster Labels:", labels_dbscan)



In [18]:
cluster_users(data)


K-Means Cluster Labels: [2 0 0 0 1]
K-Means Silhouette Score: 0.34736404635638224
DBSCAN Cluster Labels: [-1 -1 -1 -1 -1]


In [19]:
# --- 5. Predicting User Preferences ---

def predict_user_preferences(data):
    """
    Predicts user preferences (activity type).

    Args:
        data (pd.DataFrame): DataFrame containing Garmin activity data.

    Returns:
        None: Prints classification results (example).
    """

    # --- Predicting Activity Type ---
    # Features (example)
    features_activity = ['Distance', 'Avg HR', 'Total Ascent', 'Calories'] # Added 'Calories'
    data = data[features_activity + ['Activity Type']].copy()

    # Preprocessing
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Imputation
    imputer = SimpleImputer(strategy='most_frequent')
    data = imputer.fit_transform(data)

    X_activity = data[features_activity]
    y_activity = data['Activity Type']

    # Scaling
    scaler = StandardScaler()
    X_activity = scaler.fit_transform(X_activity)

    # Split data
    X_train_activity, X_test_activity, y_train_activity, y_test_activity = train_test_split(
        X_activity, y_activity, test_size=0.2, random_state=42
    )

    # --- Example: Multinomial Logistic Regression (Classification) ---
    # from sklearn.linear_model import LogisticRegression
    model_activity = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)
    model_activity.fit(X_train_activity, y_train_activity)
    y_pred_activity = model_activity.predict(X_test_activity)
    print("Activity Type Prediction Report:\n", classification_report(y_test_activity, y_pred_activity))
    print("Activity Type Prediction Score:", model_activity.score(X_test_activity, y_test_activity))



In [28]:
predict_user_preferences(data)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices